## Example

This jupyter notebook intends to provide an example on how to use the ``tovpy``. 

The following cell shows an example of how to use the ``eos.py`` to convert between pseudoenthalpy, pressure and energy density.

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.constants import c, G
import os
try:
    SCRIPTDIR = os.path.dirname(os.path.realpath(__file__))
except NameError:
    SCRIPTDIR = os.getcwd()
    # this is for interactive enviroments like Jupyter Notebooks or IPython


import sys
sys.path.append(os.path.join(SCRIPTDIR, os.pardir))
from tovpy.units import Units
uts = Units()
from tovpy.eos import EOS
from tovpy.eos import EOSPiecewisePolytropic

EoS = EOS('tabular',name=None,filename='../eos/eosG')
eos=EOSPiecewisePolytropic('SLy')#('piecewise_poly_1',gamma=5/3,K=6.560909879175998e+21)

pc = np.logspace(-19, -4, 200)
ec = np.array([EoS.EnergyDensity_Of_Pressure(pc[i]) for i in range(len(pc))])
ec2 = np.array([eos.EnergyDensity_Of_Pressure(pc[i]) for i in range(len(pc))])
plt.loglog(pc,ec,label='eosG')
plt.loglog(pc,ec2,label='SLy')
plt.legend()
plt.xlabel('Pressure')
plt.ylabel('Energy Density')
plt.show()

This cell shows a quick example of the use of ``tov.py`` to solve the TOV equations along with the calculation of Love numbers.

In [ ]:
from tovpy.tov import TOV
this_tov = TOV(eos = eos,  leven = [2,3,10], 
                                     lodd = [2], 
                                     ode_atol=1e-6, 
                                     ode_rtol=1e-6, 
                                     dhfact=-1e-12,
                                 )

fig, ax = plt.subplots(4, sharex=True)
for i, pc in enumerate(np.logspace(-11, -9, 20)):

    M,R,C,k,h,j = this_tov.solve(pc)
    R *= 1./1e3
    M *= 1./uts.constant['MRSUN_SI'][0]    
    ax[0].scatter(R, M, color='grey')
    ax[1].scatter(R, k[2], color='red')
    ax[1].scatter(R, k[3], color='green')
    ax[1].scatter(R, k[10], color='blue')
    ax[2].scatter(R, h[2], color='red')
    ax[2].scatter(R, h[3], color='green')
    ax[2].scatter(R, h[10], color='blue')
    ax[3].scatter(R, j[2], color='red')
plt.show()

In [ ]:
central_pressure = 1e-10
result = this_tov.solve(central_pressure)
sol = this_tov.sol
baryon_mass = this_tov.Compute_baryon_mass(sol)/G*c**2/2e30
proper_radius = this_tov.Compute_proper_radius(sol)/1e3

print("Baryon mass:", baryon_mass)
print("Proper radius:", proper_radius)

### Pluggable ODE solver backends

The `ode_backend` parameter lets you choose the ODE integrator used internally by `TOV.solve()`.  
Available options are:

* `'scipy'` (default) — wraps `scipy.integrate.solve_ivp`
* `'numba'` — pure-NumPy adaptive RK45 (independent of scipy, structured for future numba JIT)
* `'jax'` — JAX/diffrax Dopri5 with a fully XLA-JIT-compiled native RHS; ~3× faster than scipy 
after one-time JIT compilation warmup; requires `pip install "jax[cpu]" diffrax`.  
EOS interpolation tables are passed as dynamic arguments so that switching EOS reuses the
compiled kernel without recompilation.
* a pre-built `ODESolver` instance from `tovpy.solvers.make_solver`

The cell below demonstrates all backends, benchmarks their wall-clock time on a single pressure,
then shows the key JAX optimizations: sequence performance (varying pressures, same EOS) and
EOS-switching performance (different EOS, shared compiled kernel).


In [ ]:
import time
from tovpy.tov import TOV
from tovpy.solvers import make_solver

# --- correctness: all backends must agree ---
pc = 1e-8
M_conversion = 1./uts.constant['MRSUN_SI'][0]

tov_scipy = TOV(eos=eos, ode_backend='scipy', leven=[2], lodd=[2])
M_s, R_s, C_s, k_s, h_s, j_s = tov_scipy.solve(pc)

tov_numba = TOV(eos=eos, ode_backend='numba', leven=[2], lodd=[2])
M_n, R_n, C_n, k_n, h_n, j_n = tov_numba.solve(pc)

# --- you can also pass a pre-built solver instance ---
custom_solver = make_solver('scipy', method='RK45')
tov_custom = TOV(eos=eos, ode_backend=custom_solver, leven=[2], lodd=[2])
M_c, R_c, C_c, k_c, h_c, j_c = tov_custom.solve(pc)

try:
    tov_jax = TOV(eos=eos, ode_backend='jax', leven=[2], lodd=[2])
    # warm up the JIT compiler before timing
    M_j, R_j, C_j, k_j,h_j, j_j = tov_jax.solve(pc)
    _jax_available = True
except ImportError:
    _jax_available = False
    print('jax/diffrax not installed — skipping JAX backend')

print(f"scipy  : M={M_s*M_conversion:.6g} Msun, R={R_s/1000:.6g} km, C={C_s:.6f}, k2={k_s[2]:.6f}, j2={j_s[2]:.6f}")
print(f"numba  : M={M_n*M_conversion:.6g} Msun, R={R_n/1000:.6g} km, C={C_n:.6f}, k2={k_s[2]:.6f}, j2={j_s[2]:.6f}")
print(f"custom : M={M_c*M_conversion:.6g} Msun, R={R_c/1000:.6g} km, C={C_c:.6f}, k2={k_c[2]:.6f}, j2={j_c[2]:.6f}")
if _jax_available:
    print(f"jax    : M={M_j*M_conversion:.6g} Msun,  R={R_j/1000:.6g} km,  C={C_j:.6f}, k2={k_j[2]:.6f}, j2={j_j[2]:.6f}")
    print(f"M rel. diff (scipy vs jax): {abs(M_s - M_j) / M_s:.2e}")
print(f"M rel. diff (scipy vs numba): {abs(M_s - M_n) / M_s:.2e}")
print(f"M rel. diff (scipy vs custom): {abs(M_s - M_c) / M_s:.2e}")

# --- speed comparison: single pressure ---
N = 30
backends = [('scipy', tov_scipy), ('numba', tov_numba), ('custom', tov_custom)]
if _jax_available:
    backends.append(('jax', tov_jax))
print('\n--- Single-pressure benchmark ---')
for name, tov in backends:
    t0 = time.perf_counter()
    for _ in range(N):
        tov.solve(pc)
    elapsed = (time.perf_counter() - t0) / N * 1000
    print(f"{name:8s}: {elapsed:.2f} ms/call")

# --- speed comparison: pressure sequence (same EOS) ---
pc_array = np.logspace(-12, -9, 20)
print(f'\n--- Sequence benchmark ({len(pc_array)} pressures, same EOS) ---')
for name, tov in backends:
    t0 = time.perf_counter()
    for p in pc_array:
        tov.solve(p)
    elapsed = time.perf_counter() - t0
    print(f"{name:8s}: {elapsed:.3f}s total ({elapsed/len(pc_array)*1000:.1f} ms/solve)")

# --- speed comparison: EOS switching (JAX kernel reuse) ---
if _jax_available:
    eos_names = ['SLy', 'AP1', 'FPS', 'WFF1', 'BBB2',
                 'ENG', 'MPA1', 'MS1', 'ALF2', 'H4']
    print(f'\n--- EOS-switching benchmark ({len(eos_names)} EOS × {len(pc_array)} pressures) ---')
    for backend_name in ['scipy', 'jax']:
        total = 0
        for name in eos_names:
            eos_i = EOSPiecewisePolytropic(name)
            tov_i = TOV(eos=eos_i, ode_backend=backend_name)
            t0 = time.perf_counter()
            for p in pc_array:
                tov_i.solve(p)
            elapsed = time.perf_counter() - t0
            total += elapsed
            print(f"  {backend_name:8s} + {name:5s}: {elapsed:.3f}s ({elapsed/len(pc_array)*1000:.1f} ms/solve)")
        print(f"  {backend_name:8s} TOTAL: {total:.3f}s")


Here is to show how to use ``utils.py`` to quickly visualise the data and to save the data in ``.txt`` files. Note that ``utils.py`` cannot provide the perfect plots with full flexibility, to customise your own plot, please use the exorted ``.txt`` data.

In [ ]:
# In your Jupyter Notebook

# Import necessary packages and modules
import numpy as np
import matplotlib.pyplot as plt
import os

# Import the EOS and Units classes (assuming these are defined in the tovpy package)
from tovpy.eos import EOS
from tovpy.units import Units

# Import the Utils class from your utils module
from tovpy.utils import Utils

# Initialize the Units object (if needed)
uts = Units()

# Create an EOS instance.
# Note: Adjust the parameters for EOS() based on your actual implementation.
# eos = EOS('tabular',name="from_file",filename='eosG')  # for example, EOS(polytropic_index=...) if required

# Define an array of central pressures (example values, adjust as needed)
# Here we create 50 logarithmically spaced pressure values
p = np.logspace(-11, -9, 50)

# Specify a path where you want to save the output data/plots
save_path = "results"

# Create an instance of the Utils class using your EOS and pressure array
utils_instance = Utils(eos=eos, p=p, path=save_path)

# Now you can call the various methods:

# 1. Plot the Equation of State (EOS)
utils_instance.eos_plot(savefigon=True)
# This will display the EOS plot and also save it (using a default filename)


# 2. Save EOS data to a text file
utils_instance.eos_txt()
# The data will be saved to a file (e.g., "<EOS>eos_data.txt") in your results directory

# 3. Plot the Mass-Radius (MR) relation
utils_instance.MR_plot(savefigon=True)
# This displays and saves the mass-radius plot

# 4. Save the Mass-Radius data to a text file
utils_instance.MR_txt()

# 5. For plotting the Love numbers and moments of inertia, you need to provide
#    the "leven" and "lodd" parameters. Here are example arrays:
leven = [2, 3, 4, 9]  # even-parity multipoles (example values)
lodd  = [2, 3, 4, 9]  # odd-parity multipoles (example values)

# Plot the Love number and moment of inertia relation
utils_instance.Love_plot(leven=leven, lodd=lodd, savefigon=True)

# Save the Love data to a text file
utils_instance.Love_txt(leven=leven, lodd=lodd)

